In [ ]:
# @title Rush Neuron URL to Text or YDKE
# @markdown Enter Neuron URL:
url = "" # @param {"type":"string","placeholder":"Enter Neuron URL"}
# @markdown Use English card names from Yugipedia:
english_names = True # @param {"type":"boolean"}
# @markdown Generate YDKE URL instead:
ydke = False # @param {"type":"boolean"}
# @markdown How to import deck to EdoPro:
# @markdown 1. Copy the generated YDKE URL.
# @markdown 2. Open EdoPro and go to the deck editor.
# @markdown 3. Press CTRL+V or click "YDKE" and then "Import".

import csv
import requests
from requests.adapters import HTTPAdapter, Retry
from lxml import html
if ydke:
    import sqlite3
    import base64
    import numpy as np

stored_names={}
half2full=dict((i,i+0xFEE0) for i in range(0x21,0x7F))
def jp2en(value):
    value=value.translate(half2full)
    try:
        en=stored_names[value]
    except:
        try:
          en=list(s.get('https://yugipedia.com/api.php?action=ask&query=[[Japanese%20base%20name::'+value+']]%20AND%20[[Rush%20Duel%20status::%2B]]&format=json',headers={'User-agent':'Mozilla/5.0'}).json()['query']['results'].keys())[0]
        except:
          print('Failed to find English name for '+value)
          en=value
        en=en.replace('(','[').replace(')',']')
        if en[-12:]==' [Rush Duel]':
            en=en[:-12]
            if ydke:
                en+=' (Rush)'
        stored_names[value]=en
    return en

def getid(value):
    try:
        try:
            id=cur.execute('SELECT id FROM texts WHERE LOWER(name)=LOWER("'+value+'")').fetchone()[0]
        except:
            id=precur.execute('SELECT id FROM texts WHERE LOWERname=LOWER("'+value+'")').fetchone()[0]
        return id
    except:
        try:
            try:
                id=cur.execute('SELECT id FROM texts WHERE LOWER(name)=LOWER("'+value[:-7]+'")').fetchone()[0]
            except:
                id=precur.execute('SELECT id FROM texts WHERE LOWER(name)=LOWER("'+value[:-7]+'")').fetchone()[0]
            return id
        except:
            print('Failed to find EdoPro card ID for'+value)
            return 0

data=[]
s=requests.Session()
retries=Retry(total=100,backoff_factor=0.1,status_forcelist=[500,502,503,504])
s.mount('http://', HTTPAdapter(max_retries=retries))

page=s.get(url)
tree=html.fromstring(page.content)
mon=tree.xpath('//table[@id="monster_list"]//div[@class="icon"]/span/text()')
spell=tree.xpath('//table[@id="spell_list"]//div[@class="icon"]/span/text()')
trap=tree.xpath('//table[@id="trap_list"]//div[@class="icon"]/span/text()')
extra=tree.xpath('//table[@id="extra_list"]//div[@class="icon"]/span/text()')
side=tree.xpath('//table[@id="side_list"]//div[@class="icon"]/span/text()')
moncopy=tree.xpath('//table[@id="monster_list"]//td[@class="num"]/span/text()')
moncopy=[copy.strip() for copy in moncopy]
spellcopy=tree.xpath('//table[@id="spell_list"]//td[@class="num"]/span/text()')
spellcopy=[copy.strip() for copy in spellcopy]
trapcopy=tree.xpath('//table[@id="trap_list"]//td[@class="num"]/span/text()')
trapcopy=[copy.strip() for copy in trapcopy]
extracopy=tree.xpath('//table[@id="extra_list"]//td[@class="num"]/span/text()')
extracopy=[copy.strip() for copy in extracopy]
sidecopy=tree.xpath('//table[@id="side_list"]//td[@class="num"]/span/text()')
sidecopy=[copy.strip() for copy in sidecopy]
if english_names or ydke:
    mon=[jp2en(name) for name in mon]
    spell=[jp2en(name) for name in spell]
    trap=[jp2en(name) for name in trap]
    extra=[jp2en(name) for name in extra]
    side=[jp2en(name) for name in side]
datamon=[['Main Deck',mon[i],moncopy[i]] for i in range(len(mon))]
dataspell=[['Main Deck',spell[i],spellcopy[i]] for i in range(len(spell))]
datatrap=[['Main Deck',trap[i],trapcopy[i]] for i in range(len(trap))]
dataextra=[['Extra Deck',extra[i],extracopy[i]] for i in range(len(extra))]
dataside=[['Side Deck',side[i],sidecopy[i]] for i in range(len(side))]
data+=datamon+dataspell+datatrap+dataextra+dataside

decklist={'Main Deck':[],'Extra Deck':[],'Side Deck':[]}
for row in data:
    decklist[row[0]].append([row[2],row[1]])

textlist=''
maindeck=decklist['Main Deck']
extradeck=decklist['Extra Deck']
sidedeck=decklist['Side Deck']
if not ydke:
    maincount=str(sum([int(card[0]) for card in maindeck]))
    maintext='\n'.join([card[0]+'x '+card[1] for card in maindeck])
    textlist+='Main Deck ('+maincount+'):\n'+maintext+'\n\n'
    extracount=str(sum([int(card[0]) for card in extradeck]))
    extratext='\n'.join([card[0]+'x '+card[1] for card in extradeck])
    textlist+='Extra Deck ('+extracount+'):\n'+extratext+'\n\n'
    sidecount=str(sum([int(card[0]) for card in sidedeck]))
    sidetext='\n'.join([card[0]+'x '+card[1] for card in sidedeck])
    textlist+='Side Deck ('+sidecount+'):\n'+sidetext
    print(textlist)
else:
    try:
        cur=sql.cursor()
        precur=presql.cursor()
    except:
        !wget 'https://github.com/ProjectIgnis/BabelCDB/raw/refs/heads/master/cards-rush.cdb' -O rush.cdb -q
        !wget 'https://github.com/ProjectIgnis/BabelCDB/raw/refs/heads/master/prerelease-cards-rush.cdb' -O pre.cdb -q
        sql=sqlite3.connect('rush.cdb')
        presql=sqlite3.connect('pre.cdb')
        cur=sql.cursor()
        precur=presql.cursor()
    cardids={'Main Deck':[],'Extra Deck':[],'Side Deck':[]}
    for card in maindeck:
        cardids['Main Deck']+=[getid(card[1])]*int(card[0])
    for card in extradeck:
        cardids['Extra Deck']+=[getid(card[1])]*int(card[0])
    for card in sidedeck:
        cardids['Side Deck']+=[getid(card[1])]*int(card[0])
    mainids=np.array(cardids['Main Deck'],dtype=np.uint32)
    extraids=np.array(cardids['Extra Deck'],dtype=np.uint32)
    sideids=np.array(cardids['Side Deck'],dtype=np.uint32)
    ydke='ydke://'
    ydke+=base64.b64encode(mainids).decode()+'!'
    ydke+=base64.b64encode(extraids).decode()+'!'
    ydke+=base64.b64encode(sideids).decode()+'!'
    print(ydke)
